# Embedding Models Evaluation for French Public Tenders (BOAMP)

This notebook compares two embedding models using **LangChain** on domain-specific French public procurement clauses:
1. **Google Gemini Embeddings** (`models/gemini-embedding-001`) via `langchain-google-genai`
2. **Multilingual-E5 Base** (`intfloat/multilingual-e5-base`) via `langchain-huggingface`

### Evaluation Metrics
- **Recall@1**: Percentage of queries where the exact ground-truth clause is ranked #1.
- **Recall@3**: Percentage of queries where the ground-truth clause appears in the top 3 results.
- **Avg Latency (ms)**: Average embedding generation time per query.

In [1]:
import os
import time
from pathlib import Path
import numpy as np
from dotenv import load_dotenv

# Load .env from project root
root_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(root_dir / ".env")

assert os.getenv("GEMINI_API_KEY"), "GEMINI_API_KEY is missing in .env"
print("Environment loaded successfully.")

Environment loaded successfully.


## 1. Domain Corpus (10 Representative BOAMP Tender Clauses)

In [2]:
TENDER_CLAUSES = [
    {
        "id": "doc_1",
        "section": "Objet du march?",
        "text": "Le pr?sent march? a pour objet la refonte, l'h?bergement et la maintenance ?volutive du portail citoyen de la collectivit?."
    },
    {
        "id": "doc_2",
        "section": "Dur?e et reconduction",
        "text": "Le march? est conclu pour une dur?e ferme de 24 mois ? compter de sa notification, reconductible deux fois pour une p?riode de 12 mois."
    },
    {
        "id": "doc_3",
        "section": "Garantie financi?re",
        "text": "Une retenue de garantie de 5 % du montant initial TTC est appliqu?e sur chaque acompte, rempla?able par une garantie ? premi?re demande."
    },
    {
        "id": "doc_4",
        "section": "S?curit? et h?bergement",
        "text": "Les donn?es ? caract?re personnel doivent ?tre h?berg?es exclusivement au sein de l'Union Europ?enne sur une infrastructure qualifi?e SecNumCloud."
    },
    {
        "id": "doc_5",
        "section": "P?nalit?s de retard",
        "text": "En cas de d?passement des d?lais contractuels de livraison, une p?nalit? journali?re de 250 EUR hors taxes sera appliqu?e sans mise en demeure pr?alable."
    },
    {
        "id": "doc_6",
        "section": "Crit?res d'attribution",
        "text": "Les offres sont jug?es selon la pond?ration suivante : 60 % pour la valeur technique du m?moire et 40 % pour le prix global forfaitaire."
    },
    {
        "id": "doc_7",
        "section": "Sous-traitance",
        "text": "Le titulaire peut sous-traiter l'ex?cution d'une partie des prestations sous r?serve de d?claration pr?alable via le formulaire DC4 et d'acceptation par l'acheteur."
    },
    {
        "id": "doc_8",
        "section": "Niveaux de service (SLA)",
        "text": "Le prestataire garantit une disponibilit? mensuelle minimale de 99,9 % en heures ouvr?es avec un temps de r?tablissement maximal de 4 heures en cas d'incident critique."
    },
    {
        "id": "doc_9",
        "section": "Visite obligatoire sur site",
        "text": "Une visite pr?alable des locaux techniques est obligatoire avant la remise des offres ; l'attestation sign?e doit ?tre jointe au dossier de candidature."
    },
    {
        "id": "doc_10",
        "section": "Modalit?s de paiement",
        "text": "Le r?glement des factures s'effectue par virement administratif dans un d?lai global maximal de 30 jours ? compter de la r?ception de la demande de paiement sur Chorus Pro."
    }
]

print(f"Loaded {len(TENDER_CLAUSES)} tender clauses.")

Loaded 10 tender clauses.


## 2. Evaluation Benchmark (20 French Queries with Ground Truth)

In [3]:
EVAL_QUERIES = [
    ("Quel est l'objet principal de cet appel d'offres ?", "doc_1"),
    ("S'agit-il d'une prestation de maintenance et refonte web ?", "doc_1"),
    ("Quelle est la dur?e initiale du contrat et combien de reconductions sont pr?vues ?", "doc_2"),
    ("Combien de temps dure le march? au total avec renouvellement ?", "doc_2"),
    ("Quel est le pourcentage de la retenue de garantie financi?re ?", "doc_3"),
    ("Peut-on remplacer la retenue de garantie par une caution bancaire ?", "doc_3"),
    ("O? doivent ?tre h?berg?es les donn?es personnelles et quelle certification est exig?e ?", "doc_4"),
    ("La qualification SecNumCloud est-elle requise pour l'h?bergement ?", "doc_4"),
    ("Quel est le montant des p?nalit?s par jour de retard ?", "doc_5"),
    ("Faut-il une mise en demeure avant d'appliquer les p?nalit?s de livraison ?", "doc_5"),
    ("Comment sont pond?r?s le prix et la valeur technique dans la notation ?", "doc_6"),
    ("Quel poids repr?sente le crit?re financier dans l'?valuation des offres ?", "doc_6"),
    ("Quel formulaire faut-il fournir pour d?clarer un sous-traitant ?", "doc_7"),
    ("La sous-traitance est-elle autoris?e sur ce march? public ?", "doc_7"),
    ("Quel est le taux de disponibilit? SLA exig? et le d?lai de r?tablissement ?", "doc_8"),
    ("En combien d'heures un incident bloquant doit-il ?tre r?solu ?", "doc_8"),
    ("La visite sur site est-elle facultative ou obligatoire avant de r?pondre ?", "doc_9"),
    ("Faut-il joindre une attestation de visite dans le dossier de candidature ?", "doc_9"),
    ("Quel est le d?lai de paiement des factures et sur quelle plateforme ?", "doc_10"),
    ("Les factures doivent-elles ?tre d?pos?es sur Chorus Pro ?", "doc_10"),
]

print(f"Loaded {len(EVAL_QUERIES)} evaluation queries.")

Loaded 20 evaluation queries.


## 3. Similarity Search & Evaluation Helper

In [4]:
def cosine_similarity_matrix(query_vec: np.ndarray, doc_matrix: np.ndarray) -> np.ndarray:
    query_norm = query_vec / np.linalg.norm(query_vec)
    doc_norms = doc_matrix / np.linalg.norm(doc_matrix, axis=1, keepdims=True)
    return np.dot(doc_norms, query_norm)


def evaluate_embedding_model(
    model_name: str,
    embedder,
    doc_prefix: str = "",
    query_prefix: str = ""
) -> dict:
    doc_texts = [f"{doc_prefix}{item['section']}: {item['text']}" for item in TENDER_CLAUSES]
    doc_ids = [item["id"] for item in TENDER_CLAUSES]

    t0_docs = time.perf_counter()
    doc_vectors = np.array(embedder.embed_documents(doc_texts), dtype=np.float32)
    doc_embed_ms = (time.perf_counter() - t0_docs) * 1000

    top1_hits = 0
    top3_hits = 0
    query_latencies = []

    for query_text, expected_id in EVAL_QUERIES:
        t0_q = time.perf_counter()
        q_vec = np.array(embedder.embed_query(f"{query_prefix}{query_text}"), dtype=np.float32)
        query_latencies.append((time.perf_counter() - t0_q) * 1000)

        scores = cosine_similarity_matrix(q_vec, doc_vectors)
        ranked_indices = np.argsort(scores)[::-1]
        ranked_ids = [doc_ids[i] for i in ranked_indices]

        if ranked_ids[0] == expected_id:
            top1_hits += 1
        if expected_id in ranked_ids[:3]:
            top3_hits += 1

    total = len(EVAL_QUERIES)
    return {
        "Model": model_name,
        "Dimensions": doc_vectors.shape[1],
        "Recall@1 (%)": round((top1_hits / total) * 100, 1),
        "Recall@3 (%)": round((top3_hits / total) * 100, 1),
        "Docs Batch (ms)": round(doc_embed_ms, 1),
        "Avg Query (ms)": round(float(np.mean(query_latencies)), 1),
    }

## 4. Run Benchmark: Gemini Embeddings vs. Multilingual-E5

In [5]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings

results = []

# 1. Google Gemini Embeddings
gemini_embedder = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=os.getenv("GEMINI_API_KEY")
)
results.append(evaluate_embedding_model("Gemini (gemini-embedding-001)", gemini_embedder))

# 2. Multilingual-E5 Base (requires 'passage: ' and 'query: ' prefixes)
e5_embedder = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-base")
results.append(
    evaluate_embedding_model(
        "Multilingual-E5 Base",
        e5_embedder,
        doc_prefix="passage: ",
        query_prefix="query: "
    )
)

for r in results:
    print(r)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
c:\Users\Asus\OneDrive\Documents\GitHub\tender-ai-agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Asus\OneDrive\Documents\GitHub\tender-ai-agent\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Asus\.cache\huggingface\hub\models--intfloat--multilingual-e5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To suppor

{'Model': 'Gemini (gemini-embedding-001)', 'Dimensions': 3072, 'Recall@1 (%)': 100.0, 'Recall@3 (%)': 100.0, 'Docs Batch (ms)': 1245.2, 'Avg Query (ms)': 366.7}
{'Model': 'Multilingual-E5 Base', 'Dimensions': 768, 'Recall@1 (%)': 95.0, 'Recall@3 (%)': 100.0, 'Docs Batch (ms)': 1390.3, 'Avg Query (ms)': 153.7}


## 5. Architectural Decision

- **Selected Embedding Model for FastAPI Microservice**: `models/gemini-embedding-001` via `langchain-google-genai`.
- **Rationale**:
  1. Achieves top retrieval accuracy on French administrative & legal procurement vocabulary.
  2. Keeps the Docker container lightweight (~150 MB vs >2 GB with PyTorch/HuggingFace weights) and avoids cold-start RAM overhead.
  3. Shares the exact same `GEMINI_API_KEY` authentication and retry infrastructure already built into the service.
- **Fallback / Air-gapped Option**: `intfloat/multilingual-e5-base` remains the primary local alternative if strict offline/on-premise execution is required.